In [1]:
!rocm-smi




============================================ ROCm System Management Interface ============================================
====================================================== Concise Info ======================================================
Device  Node  IDs              Temp        Power     Partitions          SCLK    MCLK    Fan  Perf  PwrCap   VRAM%  GPU%  
              (DID,     GUID)  (Junction)  (Socket)  (Mem, Compute, ID)                                                   
0       3     0x74a5,   51110  46.0°C      119.0W    NPS1, SPX, 0        132Mhz  900Mhz  0%   auto  1000.0W  0%     0%    
1       5     0x74a5,   2987   50.0°C      126.0W    NPS1, SPX, 0        132Mhz  900Mhz  0%   auto  1000.0W  0%     0%    
2       4     0x74a5,   61326  47.0°C      125.0W    NPS1, SPX, 0        132Mhz  900Mhz  0%   auto  1000.0W  0%     0%    
3       2     0x74a5,   9091   54.0°C      128.0W    NPS1, SPX, 0        132Mhz  900Mhz  0%   auto  1000.0W  0%     0%    
4       7    

In [2]:
!pip list

Package                           Version
--------------------------------- ------------------------------------------
absl-py                           2.3.1
accelerate                        1.12.0
aiohappyeyeballs                  2.6.1
aiohttp                           3.13.2
aiosignal                         1.4.0
amdsmi                            26.1.0+5df6c765
annotated-doc                     0.0.4
annotated-types                   0.7.0
anthropic                         0.71.0
anyio                             4.12.0
astor                             0.8.1
asttokens                         3.0.1
attrs                             25.4.0
bitsandbytes                      0.49.0.dev0
blake3                            1.0.8
boto3                             1.42.4
botocore                          1.42.4
cachetools                        6.2.2
cbor2                             5.7.1
certifi                           2025.11.12
charset-normalizer                3.4.4
click        

In [3]:
import sys
import os
os.environ['HF_HOME'] = '/work1/lgarcia/pedrobpio/HF_files'
sys.path.append("..")
%load_ext autoreload
%autoreload 2

In [4]:
from src.models.qwen3 import Qwen3

model_name = 'Qwen3-14B'
t = Qwen3(model_name = f'Qwen/{model_name}', device = "cuda:0")

t.model

/work1/lgarcia/pedrobpio/decoder-lener-iob/venv-3.12/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:root:Using device: cuda:0
`torch_dtype` is deprecated! Use `dtype` instead!
/work1/lgarcia/pedrobpio/decoder-lener-iob/venv-3.12/lib64/python3.12/site-packages/torch/library.py:357: UserWarning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: flash_attn::_flash_attn_backward(Tensor dout, Tensor q, Tensor k, Tensor v, Tensor out, Tensor softmax_lse, Tensor(a6!)? dq, Tensor(a7!)? dk, Tensor(a8!)? dv, float dropout_p, float softmax_scale, bool causal, SymInt window_size_left, SymInt window_size_right, float softcap, Tensor? alibi_slopes, bool deterministic, Tensor? rng

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 5120)
    (layers): ModuleList(
      (0-39): 40 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=False)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (up_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (down_proj): Linear(in_features=17408, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((5120,), eps=1e-06)
        (post_attention_la

In [5]:
### edit tokenizer
# new_tokens = ["PESSOA", "ORGANIZACAO", "LOCAL", "TEMPO", "LEGISLACAO", "JURISPRUDENCIA",
#               "B-PESSOA", "I-PESSOA",
#               "B-ORGANIZACAO","I-ORGANIZACAO",
#               "B-LOCAL", "I-LOCAL",
#               "B-TEMPO", "I-TEMPO",
#               "B-LEGISLACAO", "I-LEGISLACAO",
#               "B-JURISPRUDENCIA", "I-JURISPRUDENCIA"]
# new_tokens_to_add = set(new_tokens) - set(t.tokenizer.vocab.keys())
# if new_tokens_to_add:
#     num_added_toks = t.tokenizer.add_tokens(list(new_tokens_to_add))
#     print(f"Added {len(new_tokens_to_add)} new tokens.")
# else:
#     print("All new tokens already exist in the vocabulary.")
# t.model.resize_token_embeddings(len(t.tokenizer))

# input_embeddings = t.model.get_input_embeddings().weight.data
# output_embeddings = t.model.get_output_embeddings().weight.data

# # Calculate mean of existing embeddings
# input_embeddings_avg = input_embeddings[:-num_added_toks].mean(dim=0, keepdim=True)
# output_embeddings_avg = output_embeddings[:-num_added_toks].mean(dim=0, keepdim=True)

# # Assign the mean value to the new token rows
# input_embeddings[-num_added_toks:] = input_embeddings_avg
# output_embeddings[-num_added_toks:] = output_embeddings_avg

In [5]:
from src.datasets.ulysses import UlyssesDataset

l = UlyssesDataset(tokenizer=t.tokenizer)

# from src.datasets.lener import LenerDataset

# l = LenerDataset(tokenizer=t.tokenizer)

In [6]:
data = l.load_dataset()
data

INFO:src.datasets.ulysses:Loading Ulysses NER dataset PL V2
INFO:src.datasets.ulysses:Loading dataset: eduagarcia/PortuLex_benchmark


INFO:src.datasets.ulysses:Dataset loaded with splits: ['train', 'validation', 'test']
INFO:src.datasets.ulysses:Processing splits: ['train', 'validation', 'test']
INFO:src.datasets.ulysses:Original columns to remove after mapping: ['idx', 'tokens', 'ner_tags']
INFO:src.datasets.ulysses:NER tag mapping created: {0: 'O', 1: 'B-DATA', 2: 'I-DATA', 3: 'B-EVENTO', 4: 'I-EVENTO', 5: 'B-FUNDAMENTO', 6: 'I-FUNDAMENTO', 7: 'B-LOCAL', 8: 'I-LOCAL', 9: 'B-ORGANIZACAO', 10: 'I-ORGANIZACAO', 11: 'B-PESSOA', 12: 'I-PESSOA', 13: 'B-PRODUTODELEI', 14: 'I-PRODUTODELEI'}
INFO:src.datasets.ulysses:Starting dataset mapping...
Map: 4542 examples [00:09, 234.36 examples/s]          
Map: 978 examples [00:02, 237.61 examples/s]         
Map: 1048 examples [00:02, 236.08 examples/s]        
INFO:src.datasets.ulysses:Dataset mapping finished.
INFO:src.datasets.ulysses:Columns after mapping: ['idx', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels', 'prompt']


DatasetDict({
    train: Dataset({
        features: ['idx', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels', 'prompt'],
        num_rows: 2271
    })
    validation: Dataset({
        features: ['idx', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels', 'prompt'],
        num_rows: 489
    })
    test: Dataset({
        features: ['idx', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels', 'prompt'],
        num_rows: 524
    })
})

In [ ]:
from src.peft_configs.lora import LoraAdapter
# PRESET = 'attention'
# PRESET = 'full_attention'
# PRESET = 'ffn'
# PRESET = 'full_attention_plus_ffn'
# PRESET = 'low_rank_16'
# PRESET = 'low_rank_32'
# PRESET = 'low_rank_64'
# PRESET = 'medium_rank_16'
# PRESET = 'medium_rank_32'
# PRESET = 'medium_rank_64'
# PRESET = 'high_rank_16'
# PRESET = 'high_rank_32'
# PRESET = 'high_rank_64'
presets = [
    'attention',
    'full_attention',
    'ffn',
    'full_attention_plus_ffn',
    'low_rank_16',
    'low_rank_32',
    'low_rank_64',
    'medium_rank_16',
    'medium_rank_32',
    'medium_rank_64',
    'high_rank_16',
    'high_rank_32',
    'high_rank_64'
]


# lora = LoraAdapter(
#     model=t.model,
#     lora_preset = PRESET
# )

# model = lora.apply_lora()
# model

In [1]:
torch.cuda.empty_cache()

NameError: name 'torch' is not defined

In [8]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch
from trl import DataCollatorForCompletionOnlyLM
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "7"
models_names = [
    # 'Qwen3-0.6B', 
                # 'Qwen3-8B', 
                # 'Qwen3-4B', 
                # 'Qwen3-1.7B', 
                'Qwen3-14B' 
]
for model_name in models_names:
    # model_name = 'Qwen3-14B'
    torch.cuda.empty_cache()
    # t = Qwen3(model_name = f'Qwen/{model_name}', device = "cuda:0")
    # Training args
    for PRESET in presets:
        print(PRESET)
        t = Qwen3(model_name = f'Qwen/{model_name}', device = "cuda:0")

        t.model
        
        lora = LoraAdapter(
        model=t.model,
        lora_preset = PRESET
        )

        model = lora.apply_lora()
        model

        checkpoint_name = f'{model_name}_{PRESET}'
        training_args = TrainingArguments(
            output_dir=f'./outputs/checkpoints/ulysses_v1/{checkpoint_name}',
            num_train_epochs=10,
            per_device_train_batch_size=4,
            # gradient_accumulation_steps=2,
            # gradient_checkpointing=True, # Consider enabling this if memory is still an issue
            lr_scheduler_type='cosine',
            logging_steps=10,
            save_strategy="epoch",
            eval_strategy="no",
            learning_rate=4e-5,
            max_grad_norm=0.01,
            weight_decay=0.01,
            warmup_ratio=0.03,
            fp16=False, # Disable this explicitly
            bf16=True,  # Enable this
            report_to="tensorboard",       # Enable TensorBoard logging
            logging_dir=f'./runs/my_logs/{checkpoint_name}_ulysses_v1',
            # group_by_length=True,
        )

        response_template = "Resposta:\n"

        response_template_ids = t.tokenizer.encode(
            response_template, 
            add_special_tokens=False
        )

        data_collator = DataCollatorForCompletionOnlyLM(
            response_template=response_template_ids,
            tokenizer=t.tokenizer
        )

        # model.enable_input_require_grads() 

        # 3. (Optional) Explicitly enable gradient checkpointing on the model instance
        # model.gradient_checkpointing_enable()

        # model.to('cuda:7')
        # torch.cuda.set_device(7)
        # Trainer

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=data["train"],
            tokenizer=t.tokenizer,
            data_collator=data_collator,
            
        )


        # Train
        trainer.train()

INFO:root:Using device: cuda:0


high_rank_32


Loading checkpoint shards: 100%|██████████| 8/8 [00:00<00:00, 532.19it/s]
INFO:root:Applying LoRA with preset: high_rank_32
INFO:root:LoRA configurations: {'r': 32, 'lora_alpha': 32, 'target_modules': 'all-linear', 'lora_dropout': 0.1, 'bias': 'none', 'task_type': <TaskType.CAUSAL_LM: 'CAUSAL_LM'>}
/tmp/ipykernel_733846/3316574100.py:75: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 128,450,560 || all params: 14,896,757,760 || trainable%: 0.8623


Step,Training Loss
10,0.447500
20,0.384300
30,0.171400
40,0.043300
50,0.019800
60,0.015600
70,0.012300
80,0.007400
90,0.008100
100,0.008600


/work1/lgarcia/pedrobpio/decoder-lener-iob/venv-3.12/lib64/python3.12/site-packages/trl/trainer/utils.py:153: UserWarning: Could not find response key `[1061, 38531, 510]` in the following instance: Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Identifica instituições, entidades administrativas ou grupos formais, como Órgãos do governo, empresas, ONGs e partidos políticos
- PESSOA: Refere-se a seres humanos mencionados nos documentos.
- DATA: Refere-se a marcações temporais presentes nos textos legislativos. como datas, horários, períodos, etc.
- LOCAL: Indica localizações geográficas ou espaços onde ocorrem os trâmites.Incluindo entidades fisicas como países, cidades, prédios e também espaços virtuais como sites oficiais
- EVENTO: Marca acontecimentos específicos que têm nome e duração definida. como Sessões plenárias, conferências, audiências públicas,
- FUNDAMENTO: Refere-se à base

In [12]:
import torch

# 1. Get your raw list of token lists
raw_prompts = data['validation'][10:25]['prompt'] 

# IMPORTANT: If this list is huge (e.g., 1000+ items), slicing it is recommended 
# to avoid Out Of Memory (OOM) errors.
# batch_prompts = raw_prompts[:8] # Example: take only the first 8
batch_prompts = raw_prompts # Use this if your batch is small enough for your GPU

# 2. Determine the Pad Token
# If your tokenizer doesn't have a pad token, use EOS
pad_token_id = t.tokenizer.pad_token_id if t.tokenizer.pad_token_id is not None else t.tokenizer.eos_token_id

# 3. Perform Manual Left-Padding
max_len = max(len(p) for p in batch_prompts)
padded_input_ids = []
attention_masks = []

for p in batch_prompts:
    # Calculate how much padding is needed for this specific sequence
    num_pads = max_len - len(p)
    
    # Create the padded row (LEFT padding)
    # [PAD, PAD, ..., TOKEN, TOKEN]
    padded_row = [pad_token_id] * num_pads + p
    
    # Create the attention mask
    # 0 for pad tokens, 1 for real tokens
    mask_row = [0] * num_pads + [1] * len(p)
    
    padded_input_ids.append(padded_row)
    attention_masks.append(mask_row)

# 4. Convert to Tensors and move to GPU
input_ids = torch.tensor(padded_input_ids).to(model.device)
attention_mask = torch.tensor(attention_masks).to(model.device)

# 5. Generation Config 
generation_config = {
    "do_sample": False,
    "temperature": 0.01,
    "top_p": 0.9,
    "repetition_penalty": 1.0, 
    "max_new_tokens": 512,
    "pad_token_id": pad_token_id,
    "eos_token_id": t.tokenizer.eos_token_id 
}

# 6. Generate for the whole batch
outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask, # Crucial so the model ignores the left-padding
    **generation_config
)

# 7. Decode the outputs
decoded_outputs = t.tokenizer.batch_decode(outputs, skip_special_tokens=True)

for text in decoded_outputs:
    print("-" * 30)
    print(text)

------------------------------
Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Identifica instituições, entidades administrativas ou grupos formais, como Órgãos do governo, empresas, ONGs e partidos políticos
- PESSOA: Refere-se a seres humanos mencionados nos documentos.
- DATA: Refere-se a marcações temporais presentes nos textos legislativos. como datas, horários, períodos, etc.
- LOCAL: Indica localizações geográficas ou espaços onde ocorrem os trâmites.Incluindo entidades fisicas como países, cidades, prédios e também espaços virtuais como sites oficiais
- EVENTO: Marca acontecimentos específicos que têm nome e duração definida. como Sessões plenárias, conferências, audiências públicas,
- FUNDAMENTO: Refere-se à base legal que sustenta o documento.  Incluindo Leis vigentes, decretos, a Constituição,etc
- PRODUTODELEI: Identifica os resultados práticos ou sistemas gerados a partir d

In [68]:


raw_prompts = data['validation']['prompt'] 
batch_prompts = raw_prompts

max_len = max(len(p) for p in batch_prompts)
padded_input_ids = []
attention_masks = []

pad_token_id = t.tokenizer.pad_token_id

for p in batch_prompts:
    # Calculate how much padding is needed for this specific sequence
    num_pads = max_len - len(p)
    
    # Create the padded row (LEFT padding)
    # [PAD, PAD, ..., TOKEN, TOKEN]
    padded_row = [pad_token_id] * num_pads + p
    
    # Create the attention mask
    # 0 for pad tokens, 1 for real tokens
    mask_row = [0] * num_pads + [1] * len(p)
    
    padded_input_ids.append(padded_row)
    attention_masks.append(mask_row)


input_ids = torch.tensor(padded_input_ids).to(model.device)
attention_mask = torch.tensor(attention_masks).to(model.device)

# input_ids = torch.tensor(token_list).to(model.device)
# inputs = data['validation']['prompt']

# 6. Gere o texto
# pad_token_id é importante para evitar avisos
generation_config = {
    "do_sample": False,          # Habilita a amostragem
    "temperature": 0.01,         # Controla a "criatividade". Mais baixo = mais focado.
    "top_p": 0.9,               # Nucleus sampling: considera tokens até somarem 90% de prob.
    "repetition_penalty": 1, # Penaliza tokens que já apareceram (valores > 1.0)
    "max_new_tokens": 1024,
    "pad_token_id": t.tokenizer.eos_token_id,
    "eos_token_id": t.tokenizer.eos_token_id # Garante que ele saiba quando parar
}

outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    **generation_config
    
)

# # 7. Decodifique a saída
# generated_text = t.tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(generated_text)

ValueError: Greedy methods without beam search do not support `num_return_sequences` different than 1 (got 32).

In [10]:
import torch
from tqdm import tqdm

# 1. Configuration
batch_size = 64  # Adjust based on your GPU VRAM
device = model.device

# Ensure we have a valid pad token
pad_token_id = t.tokenizer.pad_token_id
if pad_token_id is None:
    pad_token_id = t.tokenizer.eos_token_id

# 2. Get the list of tokenized prompts
# Assuming raw_prompts is a list of lists: [[101, 20, ...], [101, 55, ...]]
raw_prompts = data['validation']['prompt'] 

all_generated_ids = []

# 3. Iterate in batches
for i in tqdm(range(0, len(raw_prompts), batch_size), desc="Generating"):
    # Slice the current batch
    batch_sequences = raw_prompts[i : i + batch_size]
    
    # --- Dynamic Padding Logic (Applied per batch) ---
    # Find the max length needed strictly for THIS batch
    batch_max_len = max(len(seq) for seq in batch_sequences)
    
    padded_input_ids = []
    attention_masks = []
    
    for seq in batch_sequences:
        num_pads = batch_max_len - len(seq)
        
        # LEFT Padding (Crucial for generation)
        padded_row = [pad_token_id] * num_pads + seq
        
        # Attention Mask (0 for pad, 1 for real)
        mask_row = [0] * num_pads + [1] * len(seq)
        
        padded_input_ids.append(padded_row)
        attention_masks.append(mask_row)
        
    # Convert to tensors
    input_ids = torch.tensor(padded_input_ids, dtype=torch.long).to(device)
    attention_mask = torch.tensor(attention_masks, dtype=torch.long).to(device)
    
    # 4. Generate
    generation_config = {
        "do_sample": False,
        "temperature": 0.01,
        "top_p": 0.9,
        "repetition_penalty": 1.0,
        "max_new_tokens": 1024,
        "pad_token_id": pad_token_id,
        "eos_token_id": t.tokenizer.eos_token_id
    }

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **generation_config
        )
    
    # 5. Store results
    # We convert tensors back to lists to save CPU RAM, or keep as CPU tensors
    all_generated_ids.extend(outputs.cpu().tolist())

    # Optional: Clear cache if running on low-memory GPU
    # torch.cuda.empty_cache()
print(f"Generated {len(all_generated_ids)} sequences.")

# 6. Decode (Convert IDs back to text)b when needed
# You can do this here or process the IDs directly
decoded_texts = t.tokenizer.batch_decode(all_generated_ids, skip_special_tokens=True)

Generating: 100%|██████████| 5/5 [09:31<00:00, 114.31s/it]


Generated 1177 sequences.


In [70]:
torch.cuda.empty_cache()

In [11]:
# from datasets import load_dataset
from seqeval.metrics import classification_report

# 1. Load Lener-Br to get the ID-to-Label mapping
# dataset = load_dataset("lener_br", trust_remote_code=True)

# Extract the feature mapping (0 -> 'O', 1 -> 'B-ORGANIZACAO', etc.)
# Note: Lener-BR usually uses unaccented caps (ORGANIZACAO, not ORGANIZAÇÃO)
# label_list = data['train'].features['ner_tags'].feature.names
# id2label = {i: label for i, label in enumerate(label_list)}

def parse_llm_output(llm_text):
    """
    Parses lines like "Súmula:B-JURISPRUDENCIA" into a list of tags.
    """
    # def _process_output(pred_tags, lener_instance):
    #     return [lener_instance.name_to_tag_id.get(tag) for tag in pred_tags]
    
    pred_tags = []
    llm_text = llm_text.split("Resposta:\n")[-1]
    lines = llm_text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line: continue
            
        # Split by the LAST colon to separate Token from Tag
        # We use rsplit because the token itself might contain a colon (e.g., 10:30)
        if ':' in line:
            parts = line.rsplit(':', 1)
            tag = parts[-1].strip()
            pred_tags.append(tag)
        else:
            # Fallback if format is broken (LLM hallucination)
            pred_tags.append('O')
    
    return pred_tags

def align_predictions(true_len, pred_tags):
    """
    Ensures predicted tags match the length of the ground truth.
    """
    # If LLM produced too few tags, pad with 'O'
    if len(pred_tags) < true_len:
        pred_tags += ['O'] * (true_len - len(pred_tags))
        
    # If LLM produced too many tags, truncate
    elif len(pred_tags) > true_len:
        pred_tags = pred_tags[:true_len]
        
    return pred_tags

In [80]:
for i, seq in enumerate(preds):
    if '' in seq:
        print(f"Found empty tag in sequence index {i}:")
        print(seq)
        break

Found empty tag in sequence index 840:
['O', 'B-PESSOA', 'I-PESSOA', 'I-PESSOA', 'O', 'O', 'O', 'O', 'O', 'O', '', 'O', 'B-PESSOA', 'I-PESSOA', 'I-PESSOA', 'O', '', 'B-PESSOA', 'I-PESSOA', 'I-PESSOA', 'O', 'O', 'O', 'B-ORGANIZACAO', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANIZACAO', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PESSOA', 'I-PESSOA', 'I-PESSOA', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [24]:
[[l.tag_id_to_name[tag] for tag in true_tag] for true_tag in true_tags ][0]

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-JURISPRUDENCIA',
 'O',
 'B-JURISPRUDENCIA',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'I-ORGANIZACAO',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-PESSOA',
 'I-PESSOA',
 'I-PESSOA',
 'I-PESSOA',
 'I-PESSOA',
 'O',
 'O',
 'O',
 'B-PESSOA',
 'I-PESSOA',
 'I-PESSOA',
 'B-JURISPRUDENCIA',
 'I-JURISPRUDENCIA',
 'I-JURISPRUDENCIA',
 'I-JURISPRUDENCIA',
 'O',
 'O',
 'O',
 'O',
 'O']

In [25]:
clean_preds[0]

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'O',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'O',
 'O',
 'O',
 'O',
 'O',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'I-LEGISLACAO',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O']

In [12]:
# generated_text = t.tokenizer.decode(outputs, skip_special_tokens=True)
# generated
# preds = list(map(parse_llm_output, outputs))
# pred = parse_llm_output(generated_text, l)
preds=[parse_llm_output(output) for output in decoded_texts]
true_tags = data['validation']['ner_tags']
# pred = align_predictions(len(true_tags), pred)
preds = [align_predictions(len(true_tags[idx]), pred) for idx, pred in enumerate(preds)]

clean_preds = [
    ['O' if tag == '' else tag for tag in seq] 
    for seq in preds
]
print(classification_report([[l.tag_id_to_name[tag] for tag in true_tag] for true_tag in true_tags ], clean_preds))

                precision    recall  f1-score   support

JURISPRUDENCIA       0.78      0.56      0.65       207
    LEGISLACAO       0.76      0.78      0.77       397
         LOCAL       0.54      0.51      0.53       109
      OPERACAO       0.00      0.00      0.00         0
   ORGANIZACAO       0.74      0.81      0.78       561
          PESS       0.00      0.00      0.00         0
        PESSOA       0.86      0.87      0.86       310
         TEMPO       0.92      0.92      0.92       234
             _       0.00      0.00      0.00         0

     micro avg       0.78      0.78      0.78      1818
     macro avg       0.51      0.50      0.50      1818
  weighted avg       0.78      0.78      0.78      1818



/work1/lgarcia/pedrobpio/decoder-lener-iob/decoder-ie-venv/lib64/python3.9/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [25]:
true_tags

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 11,
 0,
 11,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 3,
 4,
 4,
 4,
 4,
 0,
 0,
 0,
 3,
 4,
 4,
 11,
 12,
 12,
 12,
 0,
 0,
 0,
 0,
 0]

In [11]:
from src.scripts.utils import predict_entities_batch

texts = [
    "O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula, com base na Lei 17.681/2017.",
    "- Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica ."
]

results = predict_entities_batch(texts, model, t.tokenizer)
for res in results:
    print(res)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto
O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula, com base na Lei 17.681/2017.
Resposta:
O:O
 ministério:O
 público:O
 acatou:O
 a:O
 decisão:O
 do:O
 STF:O
 e:O
 pediu:O
 a:O
 suspensão:O
 do:O
 process

In [14]:
data

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1390
    })
})

In [8]:
len(LenerDataset(tokenizer=t.tokenizer).load_dataset()['train']['input_ids'][0])

INFO:src.datasets.lener:Loading dataset: peluz/lener_br
INFO:src.datasets.lener:Dataset loaded with splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Processing splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Original columns to remove after mapping: ['id', 'tokens', 'ner_tags']
INFO:src.datasets.lener:NER tag mapping created: {0: 'O', 1: 'B-ORGANIZACAO', 2: 'I-ORGANIZACAO', 3: 'B-PESSOA', 4: 'I-PESSOA', 5: 'B-TEMPO', 6: 'I-TEMPO', 7: 'B-LOCAL', 8: 'I-LOCAL', 9: 'B-LEGISLACAO', 10: 'I-LEGISLACAO', 11: 'B-JURISPRUDENCIA', 12: 'I-JURISPRUDENCIA'}
INFO:src.datasets.lener:Starting dataset mapping...
INFO:src.datasets.lener:Dataset mapping finished.
INFO:src.datasets.lener:Columns after mapping: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels']


512

In [14]:
data['validation']['input_ids'][30]

[69286,
 3958,
 4443,
 32297,
 9087,
 16037,
 86020,
 3955,
 141402,
 4154,
 3524,
 23244,
 1197,
 13596,
 976,
 1467,
 436,
 13,
 1789,
 2121,
 1197,
 13596,
 1709,
 24709,
 36144,
 3524,
 23244,
 29610,
 1447,
 12,
 2726,
 58487,
 2843,
 135734,
 25,
 8550,
 485,
 7806,
 264,
 1197,
 13596,
 1709,
 4009,
 309,
 2872,
 17090,
 15249,
 11,
 7953,
 50304,
 11,
 7759,
 1963,
 15249,
 2569,
 2838,
 2782,
 11,
 506,
 6140,
 82,
 11,
 4992,
 624,
 12,
 393,
 9996,
 41339,
 25,
 6982,
 64,
 1197,
 13596,
 1709,
 29610,
 9662,
 288,
 409,
 45962,
 64846,
 15185,
 624,
 12,
 75670,
 2045,
 25,
 2876,
 924,
 1197,
 13596,
 1709,
 3158,
 309,
 64066,
 18965,
 2782,
 11,
 7953,
 16879,
 11,
 4812,
 37085,
 11,
 76082,
 16385,
 11,
 4992,
 624,
 12,
 42501,
 25,
 2263,
 3001,
 1197,
 13596,
 1709,
 4009,
 309,
 92824,
 3893,
 82481,
 16627,
 11,
 7953,
 272,
 13596,
 11,
 70877,
 11,
 93524,
 11,
 835,
 485,
 47919,
 11,
 4992,
 624,
 12,
 35426,
 1637,
 17845,
 74634,
 25,
 22507,
 29488,
 1197,


In [11]:
t.tokenizer.encode('''Texto: Nos termos do art . 114 , I , da Constituição da República , a Justiça do Trabalho afigura-se competente para examinar os litígios decorrentes da relação de trabalho , tenham eles fundo contratual ou não .
Resposta:
 LEGISLACAO: art . 114 , I , da Constituição da República<|im_end|>''')

[94923,
 25,
 49997,
 4647,
 436,
 653,
 1947,
 659,
 220,
 16,
 16,
 19,
 1154,
 358,
 1154,
 2994,
 75604,
 77023,
 2994,
 136962,
 1154,
 264,
 140233,
 653,
 1163,
 62794,
 6161,
 264,
 904,
 5690,
 7806,
 4533,
 6817,
 3348,
 7006,
 13762,
 2643,
 13020,
 70337,
 3530,
 10576,
 7976,
 288,
 2994,
 96103,
 409,
 54639,
 1154,
 5779,
 5604,
 66441,
 3802,
 78,
 87003,
 928,
 5908,
 12393,
 16448,
 1061,
 38531,
 510,
 35426,
 1637,
 43,
 1706,
 18746,
 25,
 1947,
 659,
 220,
 16,
 16,
 19,
 1154,
 358,
 1154,
 2994,
 75604,
 77023,
 2994,
 136962,
 151645]

In [7]:
data

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels', 'prompt'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels', 'prompt'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels', 'prompt'],
        num_rows: 1390
    })
})

In [9]:
pad_token_id = 151643
max_idx = 0
for value in data['train']:
    for idx, v in enumerate(value['input_ids']):
        if v == pad_token_id:
            if idx > max_idx:
                max_idx = idx
            break

In [10]:
max_idx

1511

In [30]:
print(t.tokenizer.decode(
    data['validation']['input_ids'][158],
    add_special_tokens=True
))

Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto
Texto: 2 Documento assinado digitalmente conforme MP n° 2.200-2/2001 de 24/08/2001 , que institui a Infraestrutura de Chaves Públicas Brasileira - ICP-Brasil .
Resposta:
2:O
Documento:O
assinado:O
digitalmente:O
conforme:O
MP:B-LEGISLACAO
n°:I-LEGISLACAO
2.20